In [1]:
import os 
import numpy as np 
import pandas as pd
%matplotlib tk
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


In [20]:
r0_true = 1000
a_true = 3.9083e-3
b_true = -5.775e-7
c_true = -4.183e-12

def model(T, r0, a, b, c):
    return r0 * (1+a* (T-273.15) + b* (T-273.15)**2 + c*( (T-273.15)-100)* (T-273.15)**3)

In [4]:
path = "C:\\Users\\MINER\\Documents\\B13 Cryolab"
file_name = os.path.join(path, "RTD.txt")

In [34]:
data = pd.read_csv(file_name, delimiter="\t")
T = data["Temperature (K)"].values
R_data = data["Resistance (Ohms)"].values 

resize = np.arange(0,np.where(T < 140)[0][0],5)
T = T[resize]
R_data = R_data[resize]

R_true = model(T, r0_true, a_true, b_true, c_true)

In [38]:
plt.scatter(T,R_data, s=1, color="C1", label="Data")
plt.scatter(T, R_true, s=1, color="C0", label="Fit")
plt.grid()
plt.xlabel("Temperature (K)")
plt.ylabel("Resistance (Ohms)")
plt.title("Resistance Curve")
plt.legend()

In [45]:
p0=[r0_true, a_true, b_true, c_true]
popt,pcov= curve_fit(model, T, R_data, p0=p0)

In [46]:
err = np.sqrt(np.diag(pcov))
for i, v in enumerate(p0):
    print(f'{popt[i]} +/- {err[i]} ({v})')

1001.854373631831 +/- 0.02619543063575764 (1000)
0.0038944697207706925 +/- 1.5288743614348379e-06 (0.0039083)
-7.96812363929862e-07 +/- 2.871485277350859e-08 (-5.775e-07)
2.597140477147295e-12 +/- 6.776920649752116e-13 (-4.183e-12)


In [64]:
samples = np.random.multivariate_normal(popt,pcov,size=5000)
ys = [model(T, *p) for p in samples]
y_lower = np.percentile(ys,0.5, axis = 0)
y_upper = np.percentile(ys,99.5, axis = 0)


In [68]:
plt.scatter(T,R_data, s=1, color="purple", label="Data")
plt.plot(T, R_true, linestyle='--', color="C1", label="Fit", alpha = 0.6)
plt.plot(T, model(T,*popt), linestyle='--', color="red", label="Model", alpha = 0.6)
plt.fill_between(T, y_lower, y_upper, color ='C0', alpha = 0.5, label="C.I (99%)")
plt.grid()
plt.xlabel("Temperature (K)")
plt.ylabel("Resistance (Ohms)")
plt.title("Resistance Curve")
plt.legend()